In [6]:
import os
import sys
import pandas as pd
from pathlib import Path

# =========================
# 1. 先建立後台資料結構
# =========================
backend_transactions = []


def clean_text(value):
    if pd.isna(value):
        return ''
    return str(value).strip()


def clean_amount(value):
    if pd.isna(value):
        return 0.0
    if isinstance(value, str):
        value = value.replace(',', '').replace(' ', '')
        if value in ['', '-', '—', 'None', 'nan', 'NaN']:
            return 0.0
    try:
        return float(value)
    except Exception:
        return 0.0


# =========================
# 2. 找到 Excel 工作簿（若有）
# =========================
def find_workbook(filename='金流工作表_0724更新後.xlsx'):
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / '表格自動化' / filename,
        Path.cwd().parent / '表格自動化' / filename,
        Path.cwd().parent / 'bdm0167' / '表格自動化' / filename,
        Path('C:/Users/BDM Intern/Desktop/bdm0167/表格自動化') / filename,
        Path('c:/Users/BDM Intern/Desktop/bdm0167/表格自動化') / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path.cwd(), Path.cwd().parent, Path('C:/Users/BDM Intern/Desktop/bdm0167'), Path('/mnt/data'), Path('/content'), Path('/workspace')]:
        if not root.exists():
            continue
        try:
            for match in root.rglob(filename):
                if match.is_file():
                    return match
        except Exception:
            pass
    return None


# =========================
# 3. 先載入你提供的範例資料（當 Excel 不存在時使用）
# =========================
def build_sample_backend():
    sample_rows = [
        {'日期': '2025-03-14', '銀行': '玉山187', '客戶/供應商': '交大', '明細': '企網非約轉帳', '支出': 5642, '收入': 0, '餘額': 2752711, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-17', '銀行': '玉山187', '客戶/供應商': '臺銀', '明細': '中華電信費用', '支出': 5500, '收入': 0, '餘額': 2747211, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-17', '銀行': '玉山187', '客戶/供應商': '臺銀', '明細': '中華電信費用', '支出': 999, '收入': 0, '餘額': 2746212, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-17', '銀行': '玉山187', '客戶/供應商': '臺銀', '明細': '中華電信費用', '支出': 4125, '收入': 0, '餘額': 2742087, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-19', '銀行': '玉山187', '客戶/供應商': '臺銀', '明細': '勞保局保險費', '支出': 1196, '收入': 0, '餘額': 2740891, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-20', '銀行': '玉山187', '客戶/供應商': '11402', '明細': '企網本行轉帳', '支出': 60000, '收入': 0, '餘額': 2680891, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-21', '銀行': '玉山187', '客戶/供應商': 'ZW27817601', '明細': '企網本行轉帳', '支出': 1780000, '收入': 0, '餘額': 900891, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-28', '銀行': '玉山187', '客戶/供應商': '兆豐銀', '明細': 'ATM跨行轉', '支出': 0, '收入': 10644, '餘額': 911535, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-28', '銀行': '玉山187', '客戶/供應商': '國世銀', '明細': 'ATM跨行轉', '支出': 0, '收入': 2000000, '餘額': 2911535, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-31', '銀行': '玉山187', '客戶/供應商': 'HB04HB05', '明細': '企網轉帳', '支出': 1121015, '收入': 0, '餘額': 1790520, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-31', '銀行': '玉山187', '客戶/供應商': '國世銀', '明細': 'ATM跨行轉', '支出': 0, '收入': 2000000, '餘額': 3790520, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-31', '銀行': '玉山187', '客戶/供應商': '國世銀', '明細': 'ATM跨行轉', '支出': 0, '收入': 200000, '餘額': 3990520, '憑證編號': '', '備註': '玉山範例', '來源': 'sample'},
        {'日期': '2025-03-04', '銀行': '台新796', '客戶/供應商': '', '明細': '代繳勞保(11401)', '支出': 27559, '收入': 0, '餘額': 48234, '憑證編號': '', '備註': '台新範例', '來源': 'sample'},
        {'日期': '2025-03-04', '銀行': '台新796', '客戶/供應商': '', '明細': '代繳勞退(11312)', '支出': 9543, '收入': 0, '餘額': 38691, '憑證編號': '', '備註': '台新範例', '來源': 'sample'},
        {'日期': '2025-03-01', '銀行': '台新809', '客戶/供應商': '', '明細': '期初餘額', '支出': 0, '收入': 0, '餘額': 113651, '憑證編號': '', '備註': '期初餘額', '來源': 'sample'},
        {'日期': '2025-03-01', '銀行': '台新854', '客戶/供應商': '', '明細': '期初餘額', '支出': 0, '收入': 0, '餘額': 27651, '憑證編號': '', '備註': '期初餘額', '來源': 'sample'},
    ]
    return pd.DataFrame(sample_rows)


# =========================
# 4. 讀取 Excel 到後台（如果有 Excel）
# =========================
def load_excel_to_backend():
    workbook_path = find_workbook()
    if workbook_path is None:
        print('⚠️ 未找到 Excel 工作簿，使用你提供的範例資料作為後台資料。')
        return build_sample_backend()

    print(f'📘 已找到 Excel：{workbook_path}')
    sheets = pd.ExcelFile(workbook_path).sheet_names
    rows = []
    for sheet in sheets:
        try:
            raw = pd.read_excel(workbook_path, sheet_name=sheet, header=None)
            header_row = None
            for idx, row in raw.iterrows():
                text = ' '.join(clean_text(x) for x in row.fillna('').tolist())
                if '帳務日期' in text and ('支出' in text or '收入' in text):
                    header_row = idx
                    break
            if header_row is None:
                continue
            data = raw.iloc[header_row + 1:].copy()
            data.columns = [clean_text(c) for c in raw.iloc[header_row].tolist()]
            for _, row in data.iterrows():
                if row.isna().all():
                    continue
                if any(str(x).strip() in {'期初餘額', '無交易'} for x in row.astype(str).tolist()):
                    continue
                try:
                    rows.append({
                        '日期': pd.to_datetime(row['帳務日期'], errors='coerce'),
                        '銀行': sheet,
                        '客戶/供應商': clean_text(row.get('客戶/供應商', '')),
                        '明細': clean_text(row.get('明細', '')),
                        '支出': clean_amount(row.get('支出', 0)),
                        '收入': clean_amount(row.get('收入', 0)),
                        '餘額': clean_amount(row.get('餘額', 0)),
                        '憑證編號': clean_text(row.get('有無憑證(憑證編號)', '')),
                        '備註': clean_text(row.get('備註', '')),
                        '來源': 'excel',
                    })
                except Exception:
                    continue
        except Exception as e:
            print(f'⚠️ 讀取工作表 {sheet} 失敗：{e}')

    if rows:
        return pd.DataFrame(rows)
    print('⚠️ Excel 內容未成功解析，使用範例資料。')
    return build_sample_backend()


# =========================
# 5. 將 Excel/範例資料寫入後台
# =========================
backend_df = load_excel_to_backend()
backend_df['日期'] = pd.to_datetime(backend_df['日期'], errors='coerce')
backend_df = backend_df.sort_values('日期').reset_index(drop=True)
backend_transactions = backend_df.to_dict(orient='records')
print(f'✅ 後台已記錄 {len(backend_transactions)} 筆交易')
backend_df.head()


# =========================
# 6. 手動輸入模式（用 input 方式追加）
# =========================
def append_manual_transactions():
    try:
        choice = input('是否要手動追加交易資料？(y/n，預設 n): ').strip().lower()
    except EOFError:
        choice = 'n'

    if choice not in {'y', 'yes'}:
        print('略過手動輸入。')
        return

    while True:
        try:
            date_str = input('請輸入日期 (YYYY-MM-DD，直接按 Enter 結束): ').strip()
        except EOFError:
            break
        if not date_str:
            break

        bank = input('請輸入銀行名稱: ').strip() or '未分類'
        customer = input('請輸入客戶/供應商: ').strip()
        detail = input('請輸入明細: ').strip()
        amount_type = input('請輸入類型 (支出/收入): ').strip().lower()
        amount = float(input('請輸入金額: ').strip() or 0)
        voucher = input('請輸入憑證編號（可留空）: ').strip()
        remark = input('請輸入備註（可留空）: ').strip()

        row = {
            '日期': pd.to_datetime(date_str, errors='coerce'),
            '銀行': bank,
            '客戶/供應商': customer,
            '明細': detail,
            '支出': amount if amount_type == '支出' else 0,
            '收入': amount if amount_type == '收入' else 0,
            '餘額': 0,
            '憑證編號': voucher,
            '備註': remark,
            '來源': 'input',
        }
        backend_transactions.append(row)
        print(f'✅ 已加入後台：{row}')

    global backend_df
    backend_df = pd.DataFrame(backend_transactions)
    backend_df['日期'] = pd.to_datetime(backend_df['日期'], errors='coerce')
    backend_df = backend_df.sort_values('日期').reset_index(drop=True)
    print(f'🧠 目前後台共有 {len(backend_df)} 筆交易')


append_manual_transactions()


# =========================
# 7. 依照後台資料建立會計分錄與報表
# =========================
def build_journal(df):
    rows = []
    for _, row in df.iterrows():
        if row['支出'] > 0:
            rows.append({
                '日期': row['日期'],
                '摘要': row['明細'] or row['客戶/供應商'] or '未註明',
                '銀行': row['銀行'],
                '借方科目': '營業費用',
                '借方金額': row['支出'],
                '貸方科目': '銀行存款',
                '貸方金額': row['支出'],
                '憑證編號': row.get('憑證編號', ''),
                '憑證狀態': '已對應' if row.get('憑證編號', '') else '待補',
                '來源': row.get('來源', ''),
            })
        elif row['收入'] > 0:
            rows.append({
                '日期': row['日期'],
                '摘要': row['明細'] or row['客戶/供應商'] or '未註明',
                '銀行': row['銀行'],
                '借方科目': '銀行存款',
                '借方金額': row['收入'],
                '貸方科目': '營業收入',
                '貸方金額': row['收入'],
                '憑證編號': row.get('憑證編號', ''),
                '憑證狀態': '已對應' if row.get('憑證編號', '') else '待補',
                '來源': row.get('來源', ''),
            })
    return pd.DataFrame(rows)


journal = build_journal(backend_df)
if journal.empty:
    journal = pd.DataFrame(columns=['日期','摘要','銀行','借方科目','借方金額','貸方科目','貸方金額','憑證編號','憑證狀態','來源'])

revenue_total = journal['貸方金額'].sum() if '貸方金額' in journal.columns else 0
expense_total = journal['借方金額'].sum() if '借方金額' in journal.columns else 0
net_profit = revenue_total - expense_total

income_statement = pd.DataFrame([
    {'項目': '營業收入', '金額': revenue_total},
    {'項目': '營業費用', '金額': -expense_total},
    {'項目': '本期淨利', '金額': net_profit},
])

cash_summary = pd.DataFrame([
    {'銀行': bank, '期末餘額': 0} for bank in backend_df['銀行'].dropna().unique()
])

balance_sheet = pd.DataFrame([
    {'類別': '資產', '科目': '現金及銀行存款', '金額': 0},
    {'類別': '資產', '科目': '流動資產合計', '金額': 0},
    {'類別': '權益', '科目': '本期淨利', '金額': net_profit},
    {'類別': '權益', '科目': '權益合計', '金額': net_profit},
])

cash_flow = pd.DataFrame([
    {'項目': '營業活動現金流量', '金額': net_profit},
    {'項目': '投資活動現金流量', '金額': 0},
    {'項目': '融資活動現金流量', '金額': 0},
    {'項目': '淨現金增加額', '金額': net_profit},
])

equity_change = pd.DataFrame([
    {'項目': '期初權益', '金額': 0},
    {'項目': '本期淨利', '金額': net_profit},
    {'項目': '期末權益', '金額': net_profit},
])

liquidity = pd.DataFrame([
    {'項目': '現金及銀行存款', '金額': 0},
    {'項目': '應收帳款', '金額': 0},
    {'項目': '存貨', '金額': 0},
    {'項目': '流動資產合計', '金額': 0},
])


# =========================
# 8. 輸出 Excel 報表
# =========================
out_path = Path.cwd() / '會計系統雛型.xlsx'
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    backend_df.to_excel(writer, sheet_name='交易主檔', index=False)
    journal.to_excel(writer, sheet_name='分錄', index=False)
    cash_summary.to_excel(writer, sheet_name='銀行餘額', index=False)
    income_statement.to_excel(writer, sheet_name='損益表', index=False)
    balance_sheet.to_excel(writer, sheet_name='資產負債表', index=False)
    cash_flow.to_excel(writer, sheet_name='現金流量表', index=False)
    equity_change.to_excel(writer, sheet_name='權益變動表', index=False)
    liquidity.to_excel(writer, sheet_name='流動資產分析', index=False)

print(f'✅ 已輸出報表：{out_path}')
print('📊 目前概況：')
print(f'  - 後台交易筆數：{len(backend_df)}')
print(f'  - 總收入：{revenue_total:,.0f}')
print(f'  - 總支出：{expense_total:,.0f}')
print(f'  - 本期淨利：{net_profit:,.0f}')


⚠️ 未找到 Excel 工作簿，使用你提供的範例資料作為後台資料。
✅ 後台已記錄 16 筆交易
略過手動輸入。
✅ 已輸出報表：/content/會計系統雛型.xlsx
📊 目前概況：
  - 後台交易筆數：16
  - 總收入：7,226,223
  - 總支出：7,226,223
  - 本期淨利：0
